# Synthetic Notebook Interface Example

This notebook is the smallest useful Modelwright notebook workflow. It starts from a tiny generated-model-shaped Python function, wraps it with `ModelFacade`, and then uses `modelwright.notebooks` helpers to display the model as tidy pandas DataFrames.

The point is not the toy arithmetic. The point is the workflow: a generated model becomes something a human can inspect, change, recalculate, and compare in a live notebook.

## 1. Make The Source Checkout Importable

If this notebook is opened from `tmp/notebooks/` or another working directory inside the checkout, Python may not automatically see the repo-root `examples/` directory. This setup cell walks upward until it finds `pyproject.toml`, then puts that directory on `sys.path`.

Expected output: the package project name, `modelwright`.

In [1]:
from pathlib import Path
import sys
import tomllib

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

if not (repo_root / "pyproject.toml").exists():
    raise RuntimeError("Could not find the Modelwright repository root.")

project_name = tomllib.loads((repo_root / "pyproject.toml").read_text())["project"]["name"]

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Using project: {project_name}")

Using project: modelwright


## 2. Load The Example Facade

`build_facade()` creates a `ModelFacade` around a tiny generated-model-shaped `calculate(inputs=None)` function. The facade gives human names to spreadsheet-style cell references and declares one rectangular table view.

Expected output: the declared input and output names.

In [2]:
from examples.synthetic.notebook_interface import build_facade
from modelwright.notebooks import compare_scenarios_frame, inputs_frame, outputs_frame, table_frame

facade = build_facade()

print(f"Declared inputs: {list(facade.inputs())}")
print(f"Declared outputs: {list(facade.outputs())}")

Declared inputs: ['base', 'growth']
Declared outputs: ['projected', 'status']


## 3. Create A Baseline And A Scenario

Scenarios are immutable input override sets. Here the baseline uses a base volume of 100 and a growth rate of 0.1. The shock scenario changes only the base volume to 120.

Expected output: the exact input overrides that will be sent to the generated model.

In [3]:
baseline = facade.scenario(name="baseline", inputs={"Inputs!B2": 100, "Inputs!B3": 0.1})
shock = baseline.with_input("Inputs!B2", 120)

print(f"Baseline inputs: {baseline.inputs}")
print(f"Shock inputs: {shock.inputs}")

Baseline inputs: {'Inputs!B2': 100, 'Inputs!B3': 0.1}
Shock inputs: {'Inputs!B2': 120, 'Inputs!B3': 0.1}


## 4. Inspect Scenario Inputs As A DataFrame

The notebook layer turns declared inputs into a DataFrame. The raw workbook references remain visible for provenance, but the human names and units are now the primary view.

Expected output: two rows, one for `base` and one for `growth`.

In [4]:
inputs_frame(facade, shock)[["name", "cell_ref", "value", "unit"]]

     name   cell_ref  value      unit
0    base  Inputs!B2  120.0         t
1  growth  Inputs!B3    0.1  fraction

## 5. Recalculate And Inspect Outputs

This cell calculates the shock scenario and displays declared outputs. The calculated projected volume is 132.0.

Expected output: `projected = 132.0` and `status = ok`.

In [5]:
outputs_frame(facade, shock)[["name", "cell_ref", "value", "unit"]]

        name    cell_ref  value unit
0  projected  Summary!B2  132.0    t
1     status  Summary!B3     ok  NaN

## 6. Display A Declared Table

The facade declared a rectangular table over `Summary!B2:C3`. The notebook helper displays it with row and column labels instead of requiring the user to reconstruct a table from cell references.

Expected output: a two-by-two DataFrame with `volume` and `status` rows.

In [6]:
table_frame(facade, "summary_grid", shock)

       primary  secondary
row                      
volume   132.0        240
status      ok        125

## 7. Compare Baseline Against Scenario

Scenario comparison is the core notebook loop. The baseline projected volume is 110.0 and the shock projected volume is 132.0, so the absolute change is 22.0 and the percent change is 0.2.

Expected output: a tidy comparison table.

In [7]:
compare_scenarios_frame(facade, baseline, shock)[
    ["name", "baseline_value", "scenario_value", "absolute_change", "percent_change"]
]

        name baseline_value scenario_value absolute_change percent_change
0  projected          110.0          132.0            22.0            0.2
1     status             ok             ok            None           None